# Complete Guide to Conformal Prediction

Comprehensive tutorial on prediction sets with statistical coverage guarantees.

**What you'll learn:**
- What conformal prediction is and why it matters
- 4 classification methods: ICP, APS, RAPS, Mondrian
- 3 regression methods: Jackknife+, CV+, CQR
- Built-in metrics and visualizations
- The `ConformalPredictor` convenience wrapper

**Runtime:** ~3 min (MPS/CUDA), ~8 min (CPU)

**Prerequisites:** Basic PyTorch knowledge

## Setup

In [ ]:
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Classification methods
from incerto.conformal import (
    inductive_conformal,
    aps,
    raps,
    mondrian_conformal,
    ConformalPredictor,
)

# Regression methods
from incerto.conformal import (
    cv_plus,
    conformalized_quantile_regression,
)

# Metrics and visualization
from incerto.conformal import (
    empirical_coverage,
    average_set_size,
    conditional_coverage,
    plot_coverage_vs_alpha,
    plot_set_size_hist,
)

from incerto.utils import ConvNet, seed_everything

seed_everything(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Part 1: What is Conformal Prediction?

Standard classifiers output a single prediction and a confidence score. But confidence
scores are often **miscalibrated** -- they don't reflect true probabilities.

**Conformal prediction** takes a different approach:
- Instead of one answer, return a **set of plausible labels**
- **Statistical guarantee:** P(true label in set) >= 1 - alpha
- **Distribution-free** -- works for any model, any data distribution

| Concept | Meaning |
|---------|---------|
| alpha | Desired error rate (e.g., 0.1 for 90% coverage) |
| Prediction set C(x) | Set of labels that might be correct |
| Coverage guarantee | P(y in C(x)) >= 1 - alpha |

**Example:** With alpha = 0.1, a medical diagnosis system returns {benign, cancer}
instead of just {benign}. The true diagnosis is in the set >= 90% of the time.

## Part 2: Load Data and Train Model

We use **Fashion-MNIST** -- a harder drop-in replacement for MNIST with 10
clothing categories. Unlike MNIST (~99% accuracy), Fashion-MNIST reaches ~90%,
so prediction sets will have interesting sizes (often 2-3 classes) and we can
see where the model is genuinely uncertain.

In [ ]:
# FashionMNIST class names for readable output
CLASS_NAMES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Load Fashion-MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, transform=transform)

# Split: 50k train, 10k calibration (CRITICAL: calibration must be separate!)
train_subset, cal_dataset = random_split(train_dataset, [50000, 10000])

# Optimized data loaders
num_workers = min(4, os.cpu_count() or 0)
pin_memory = device.type == "cuda"
loader_kwargs = dict(num_workers=num_workers, pin_memory=pin_memory,
                     persistent_workers=num_workers > 0)

train_loader = DataLoader(train_subset, batch_size=256, shuffle=True, **loader_kwargs)
cal_loader = DataLoader(cal_dataset, batch_size=512, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, **loader_kwargs)


# Helper: wrap a DataLoader so batches are moved to device automatically.
# Conformal methods iterate over the loader internally, so the data must be on
# the same device as the model.
class DeviceLoader:
    def __init__(self, loader, device):
        self.loader = loader
        self.device = device

    def __iter__(self):
        for x, y in self.loader:
            yield x.to(self.device), y.to(self.device)

    def __len__(self):
        return len(self.loader)


cal_loader_dev = DeviceLoader(cal_loader, device)

print(f"Training: {len(train_subset)} | Calibration: {len(cal_dataset)} | Test: {len(test_dataset)}")
print(f"DataLoader: num_workers={num_workers}, pin_memory={pin_memory}")

In [ ]:
# Train CNN (no dropout -- produces overconfident predictions, interesting for conformal)
model = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model.train()
print("Training (5 epochs on Fashion-MNIST)...")
for epoch in range(5):
    total_loss, correct, total = 0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Accuracy = {acc:.2f}%")

print("Done!")

## Part 3: Inductive Conformal Prediction (ICP)

The simplest conformal method. Uses the **softmax score** as the nonconformity
measure: `score = 1 - P(true class)`.

Steps:
1. Compute scores on the calibration set
2. Find the quantile threshold at level `(1-alpha)(1 + 1/n)`
3. At test time, include all classes with `P(class) >= 1 - threshold`

In [ ]:
alpha = 0.1  # 90% target coverage

# Calibrate ICP on held-out calibration set
predictor_icp = inductive_conformal(model, cal_loader_dev, alpha=alpha)

# Evaluate on test set
all_sets_icp = []
all_labels = []

for inputs, labels in test_loader:
    sets = predictor_icp(inputs.to(device))
    all_sets_icp.extend([s.cpu() for s in sets])
    all_labels.extend(labels.tolist())

labels_tensor = torch.tensor(all_labels)

coverage_icp = empirical_coverage(labels_tensor, all_sets_icp)
avg_size_icp = average_set_size(all_sets_icp)

print(f"ICP Coverage: {coverage_icp:.4f} (target >= {1-alpha:.2f})")
print(f"ICP Avg Set Size: {avg_size_icp:.2f}")

## Part 4: Adaptive Prediction Sets (APS)

APS (Romano et al., NeurIPS 2020) orders classes by probability and accumulates
until the cumulative mass exceeds the threshold. Produces **smaller sets** than
ICP because it adapts to the model's confidence for each input.

In [ ]:
predictor_aps = aps(model, cal_loader_dev, alpha=alpha)

all_sets_aps = []
for inputs, _ in test_loader:
    sets = predictor_aps(inputs.to(device))
    all_sets_aps.extend([s.cpu() for s in sets])

coverage_aps = empirical_coverage(labels_tensor, all_sets_aps)
avg_size_aps = average_set_size(all_sets_aps)

print(f"APS Coverage: {coverage_aps:.4f}")
print(f"APS Avg Set Size: {avg_size_aps:.2f}")

## Part 5: Regularized APS (RAPS)

RAPS (Angelopoulos et al., ICLR 2021) adds a regularization penalty beyond
rank `k_reg`, discouraging inclusion of low-probability classes. Also enforces
a minimum set size of `k_reg`.

Parameters:
- `lam`: Regularization strength (larger = smaller sets, but tighter threshold)
- `k_reg`: Rank beyond which penalty kicks in (also minimum set size)

In [ ]:
predictor_raps = raps(model, cal_loader_dev, alpha=alpha, lam=0.01, k_reg=2)

all_sets_raps = []
for inputs, _ in test_loader:
    sets = predictor_raps(inputs.to(device))
    all_sets_raps.extend([s.cpu() for s in sets])

coverage_raps = empirical_coverage(labels_tensor, all_sets_raps)
avg_size_raps = average_set_size(all_sets_raps)

print(f"RAPS Coverage: {coverage_raps:.4f}")
print(f"RAPS Avg Set Size: {avg_size_raps:.2f}")

## Part 6: Mondrian Conformal (Class-Conditional Coverage)

Standard conformal guarantees *marginal* coverage -- averaged over all inputs.
Mondrian conformal prediction (Papadopoulos, 2008) provides *conditional*
coverage within each partition cell (e.g., per class).

This is critical when coverage must hold per subgroup, not just on average.

In [ ]:
predictor_mond = mondrian_conformal(model, cal_loader_dev, alpha=alpha)

all_sets_mond = []
for inputs, _ in test_loader:
    sets = predictor_mond(inputs.to(device))
    all_sets_mond.extend([s.cpu() for s in sets])

coverage_mond = empirical_coverage(labels_tensor, all_sets_mond)
avg_size_mond = average_set_size(all_sets_mond)

# Per-class coverage using conditional_coverage
cond_cov = conditional_coverage(labels_tensor, all_sets_mond, groups=labels_tensor)

print(f"Mondrian Coverage: {coverage_mond:.4f}")
print(f"Mondrian Avg Set Size: {avg_size_mond:.2f}")
print(f"\nPer-class coverage (target >= {1-alpha:.2f}):")
for cls_id in sorted(cond_cov):
    print(f"  {CLASS_NAMES[cls_id]:10s}: {cond_cov[cls_id]:.4f}")

## Part 7: Compare All Methods

In [ ]:
methods = ["ICP", "APS", "RAPS", "Mondrian"]
coverages = [coverage_icp, coverage_aps, coverage_raps, coverage_mond]
avg_sizes = [avg_size_icp, avg_size_aps, avg_size_raps, avg_size_mond]

print("=" * 50)
print(f"{'Method':12s} {'Coverage':>10s} {'Avg Size':>10s}")
print("-" * 50)
for m, c, s in zip(methods, coverages, avg_sizes):
    print(f"{m:12s} {c:10.4f} {s:10.2f}")
print("=" * 50)

# Bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(methods, coverages, alpha=0.8)
ax1.axhline(y=1-alpha, color='r', linestyle='--', label=f'Target (1-alpha={1-alpha})')
ax1.set_ylabel('Empirical Coverage')
ax1.set_title('Coverage (higher is better)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2.bar(methods, avg_sizes, alpha=0.8, color='green')
ax2.set_ylabel('Average Set Size')
ax2.set_title('Efficiency (lower is better)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 8: Coverage Guarantee Analysis

The defining property of conformal prediction: coverage >= 1-alpha for any
alpha in (0, 1). Let's verify this across a range of alpha values using the
built-in `plot_coverage_vs_alpha` and `plot_set_size_hist` visualizations.

In [ ]:
# Sweep alpha from 0.01 to 0.3 and measure actual coverage
alphas = np.linspace(0.01, 0.3, 15).tolist()
coverages_sweep = []
sizes_sweep = []

print("Sweeping alpha values...")
for a in alphas:
    pred = aps(model, cal_loader_dev, alpha=a)
    sets_a = []
    for inputs, _ in test_loader:
        sets_a.extend([s.cpu() for s in pred(inputs.to(device))])
    coverages_sweep.append(empirical_coverage(labels_tensor, sets_a))
    sizes_sweep.append(average_set_size(sets_a))
print("Done!")

# Built-in visualization: coverage should track above the 1-alpha line
plot_coverage_vs_alpha(alphas, coverages_sweep)

In [ ]:
# Built-in visualization: distribution of prediction set sizes
plot_set_size_hist(all_sets_aps)

## Part 9: Visualize Prediction Sets

See which classes the model is uncertain about -- sets larger than 1 indicate
genuine ambiguity (e.g., Shirt vs Pullover, Sneaker vs Ankle boot).

In [ ]:
# Get one batch of test images
test_batch, test_labels_batch = next(iter(test_loader))
sets_vis = [s.cpu() for s in predictor_aps(test_batch.to(device))]

# Select 16 images with non-empty prediction sets for display
# (randomized APS can produce empty sets for the ~alpha fraction it abstains on)
display_indices = [i for i, s in enumerate(sets_vis) if len(s) > 0][:16]

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for plot_idx, data_idx in enumerate(display_indices):
    ax = axes[plot_idx // 4, plot_idx % 4]
    # Undo normalization for display
    img = test_batch[data_idx].squeeze() * 0.3530 + 0.2860
    ax.imshow(img.clamp(0, 1), cmap='gray')

    true_cls = test_labels_batch[data_idx].item()
    pred_set = sorted(sets_vis[data_idx].tolist())
    set_names = [CLASS_NAMES[c] for c in pred_set]
    covered = true_cls in pred_set
    color = 'green' if covered else 'red'

    ax.set_title(f"True: {CLASS_NAMES[true_cls]}\nSet: {set_names}",
                 fontsize=8, color=color)
    ax.axis('off')

plt.suptitle("APS Prediction Sets (green = covered, red = missed)", fontsize=13)
plt.tight_layout()
plt.show()

## Part 10: Conformal Regression

For regression, conformal methods produce **prediction intervals** instead of sets.

| Method | Approach | Pros |
|--------|----------|------|
| `jackknife_plus` | Leave-one-out residuals | No data splitting needed |
| `cv_plus` | K-fold cross-validation residuals | Less pessimistic than Jackknife+ |
| `conformalized_quantile_regression` (CQR) | Adjusts quantile regression | Adaptive interval width |

We demonstrate CV+ and CQR on a synthetic problem: `y = sin(2x) + noise`.

In [ ]:
# Synthetic regression dataset
torch.manual_seed(42)
n_data = 100
X_reg = torch.randn(n_data, 1) * 2
y_reg = torch.sin(2 * X_reg.squeeze()) + 0.3 * torch.randn(n_data)

n_train = 80
train_reg = TensorDataset(X_reg[:n_train], y_reg[:n_train])


# model_fn for cv_plus: takes a Dataset, returns a trained model.
# Must use torch.enable_grad() because cv_plus is wrapped in @torch.no_grad().
def train_regression_model(dataset):
    mdl = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1))
    loader = DataLoader(dataset, batch_size=len(dataset))
    opt = torch.optim.Adam(mdl.parameters(), lr=0.01)
    mdl.train()
    with torch.enable_grad():
        for _ in range(100):
            for xb, yb in loader:
                opt.zero_grad()
                loss = F.mse_loss(mdl(xb).squeeze(), yb)
                loss.backward()
                opt.step()
    mdl.eval()
    return mdl


alpha_reg = 0.1

# --- CV+ ---
print("Running CV+ (5 folds)...")
pred_cvp = cv_plus(train_regression_model, train_reg, folds=5, alpha=alpha_reg)
X_test_reg = torch.linspace(-4, 4, 200).unsqueeze(1)
lower_cvp, upper_cvp = pred_cvp(X_test_reg)
print("Done!")


# --- CQR ---
class QuantileNet(nn.Module):
    """Outputs (lower_quantile, upper_quantile) predictions."""
    def __init__(self):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(1, 32), nn.ReLU())
        self.head = nn.Linear(32, 2)

    def forward(self, x):
        return self.head(self.shared(x))


def pinball_loss(preds, targets, alpha):
    tau_lo, tau_hi = alpha / 2, 1 - alpha / 2
    err_lo = targets - preds[:, 0]
    err_hi = targets - preds[:, 1]
    loss_lo = torch.max(tau_lo * err_lo, (tau_lo - 1) * err_lo)
    loss_hi = torch.max(tau_hi * err_hi, (tau_hi - 1) * err_hi)
    return (loss_lo + loss_hi).mean()


# Train quantile model on training portion
qr_model = QuantileNet()
qr_opt = torch.optim.Adam(qr_model.parameters(), lr=0.01)
qr_loader = DataLoader(TensorDataset(X_reg[:n_train], y_reg[:n_train]),
                        batch_size=64, shuffle=True)
qr_model.train()
for _ in range(200):
    for xb, yb in qr_loader:
        qr_opt.zero_grad()
        loss = pinball_loss(qr_model(xb), yb, alpha_reg)
        loss.backward()
        qr_opt.step()

# Calibrate CQR on held-out portion
cal_reg_loader = DataLoader(
    TensorDataset(X_reg[n_train:], y_reg[n_train:]), batch_size=64
)
pred_cqr = conformalized_quantile_regression(qr_model, cal_reg_loader, alpha=alpha_reg)
lower_cqr, upper_cqr = pred_cqr(X_test_reg)
print("CQR calibrated!")

In [ ]:
# Visualize regression intervals
y_true = torch.sin(2 * X_test_reg.squeeze())
order = X_test_reg[:, 0].argsort()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, lower, upper, name, color in [
    (ax1, lower_cvp, upper_cvp, "CV+", "steelblue"),
    (ax2, lower_cqr, upper_cqr, "CQR", "darkorange"),
]:
    ax.scatter(X_reg[:n_train, 0], y_reg[:n_train], s=12, alpha=0.5, label='Train data')
    ax.plot(X_test_reg[order, 0], y_true[order], 'k-', linewidth=1.5, label='True function')
    ax.fill_between(
        X_test_reg[order, 0].numpy(),
        lower[order].detach().numpy(),
        upper[order].detach().numpy(),
        alpha=0.3, color=color,
        label=f'{name} interval (alpha={alpha_reg})',
    )
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{name} Prediction Intervals')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 11: ConformalPredictor Wrapper

`ConformalPredictor` provides an object-oriented interface that wraps any
conformal method. Use `from_method()` to create a predictor with a consistent
`.predict()` API.

In [ ]:
# Create via factory
cp = ConformalPredictor.from_method(
    "aps", model=model, calib_loader=cal_loader_dev, alpha=0.1
)
print(cp)

# Use .predict() or __call__
example_batch = test_batch[:5].to(device)
pred_sets = cp.predict(example_batch)

print("\nExample predictions:")
for i, (ps, true) in enumerate(zip(pred_sets, test_labels_batch[:5])):
    set_names = [CLASS_NAMES[c] for c in ps.cpu().tolist()]
    set_str = ", ".join(set_names)
    print(f"  Sample {i+1}: {set_str:30s} | True: {CLASS_NAMES[true.item()]}")

## Summary

### Method Selection Guide

| Situation | Recommended Method |
|-----------|--------------------|
| Default / start here | `aps` |
| Want smallest sets | `raps` with tuned `lam`/`k_reg` |
| Need per-class coverage | `mondrian_conformal` |
| Regression (simple) | `cv_plus` (or `jackknife_plus` for LOO) |
| Regression (adaptive widths) | `conformalized_quantile_regression` |
| Object-oriented interface | `ConformalPredictor.from_method()` |

### Best Practices

1. **Separate calibration data** -- never calibrate on training data
2. **Use enough calibration samples** -- at least 500, ideally 5-10% of training data
3. **Don't tune alpha on test data** -- choose alpha based on requirements
4. **Monitor set sizes** -- smaller is better (more informative)
5. **Combine with model calibration** -- well-calibrated models produce smaller sets

### Coverage Guarantee

The guarantee P(y in C(x)) >= 1-alpha holds:
- For any model (no assumptions on model quality)
- For any data distribution (distribution-free)
- In finite samples (not just asymptotically)
- As long as calibration and test data are exchangeable (i.i.d.)

### Choosing alpha

| Setting | alpha | Coverage |
|---------|-------|----------|
| Safety-critical (medical, autonomous) | 0.01--0.05 | 95--99% |
| Standard | 0.1 | 90% |
| Exploratory | 0.2 | 80% |